In [3]:
import os
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter

def rechunk_jsonl_files(input_folder, output_folder, chunk_size=500, chunk_overlap=50):
    os.makedirs(output_folder, exist_ok=True)
    
    if not os.path.exists(input_folder):
        print(f"입력 경로를 찾을 수 없습니다: {input_folder}")
        return
        
    files = [f for f in os.listdir(input_folder) if f.endswith('.jsonl')]
    if not files:
        print("입력 폴더 내에 처리할 .jsonl 파일이 없습니다.")
        return

    # 텍스트 스플리터 설정 (separators를 제거하여 500자 제한 시 강제 슬라이싱 보장)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )

    print(f"🔄 [긴급 복구] JSONL 재청킹 및 완벽 파싱 프로세스 시작")
    print("-" * 60)

    for file_name in files:
        input_path = os.path.join(input_folder, file_name)
        output_path = os.path.join(output_folder, file_name)
        
        new_chunks_count = 0
        
        try:
            with open(input_path, 'r', encoding='utf-8') as infile, \
                 open(output_path, 'w', encoding='utf-8') as outfile:
                
                for line_idx, line in enumerate(infile, 1):
                    line = line.strip()
                    if not line:
                        continue
                        
                    # 1. 일단 JSON 유효성 검사 및 로드
                    try:
                        item = json.loads(line)
                    except json.JSONDecodeError:
                        print(f"⚠️ {file_name}의 {line_idx}번째 줄은 올바른 JSON 형식이 아닙니다. 건너뜁니다.")
                        continue
                    
                    # 2. 리스트 구조 방어 ([{...}] 형태로 저장되어 있을 경우 꺼내기)
                    if isinstance(item, list) and len(item) > 0:
                        item = item[0]
                    
                    if not isinstance(item, dict):
                        continue

                    # 3. 다양한 형태의 본문 Key 추적 및 매핑
                    base_content = None
                    base_metadata = {}

                    # Case A: langchain Document 정석 구조 혹은 일반 딕셔너리
                    if 'page_content' in item:
                        base_content = item['page_content']
                        if 'metadata' in item and isinstance(item['metadata'], dict):
                            base_metadata = item['metadata']
                        else:
                            base_metadata = {k: v for k, v in item.items() if k != 'page_content'}
                    
                    # Case B: 전처리 중 text나 content라는 이름으로 유실되었을 경우 방어
                    elif 'text' in item:
                        base_content = item['text']
                        base_metadata = {k: v for k, v in item.items() if k != 'text'}
                    elif 'content' in item:
                        base_content = item['content']
                        base_metadata = {k: v for k, v in item.items() if k != 'content'}
                    
                    # 본문 텍스트를 찾지 못했다면 해당 줄 패스
                    if not base_content:
                        continue

                    # 4. 텍스트 분할 및 저장
                    split_texts = text_splitter.split_text(str(base_content))
                    
                    for idx, text in enumerate(split_texts):
                        chunk_metadata = base_metadata.copy()
                        chunk_metadata["chunk_index"] = idx
                        
                        chunk_data = {
                            "page_content": text,
                            "metadata": chunk_metadata
                        }
                        outfile.write(json.dumps(chunk_data, ensure_ascii=False) + "\n")
                        new_chunks_count += 1
                            
            print(f"✅ 완료: {file_name} -> 새 청크 {new_chunks_count}개 생성 완료")
            
        except Exception as e:
            print(f"❌ 에러 발생 ({file_name}): {e}")

    print("-" * 60)
    print(f"🎉 모든 파일 복구 및 청킹 완료! {output_folder} 폴더를 확인해 보세요.")

# --- 실행부 ---
input_dir = "../data/03_processed/new"
output_dir = "../data/04_chunks/final"

rechunk_jsonl_files(input_dir, output_dir, chunk_size=500, chunk_overlap=50)

🔄 [긴급 복구] JSONL 재청킹 및 완벽 파싱 프로세스 시작
------------------------------------------------------------
✅ 완료: kb_chunks_eflaw.jsonl -> 새 청크 2888개 생성 완료
------------------------------------------------------------
🎉 모든 파일 복구 및 청킹 완료! ../data/04_chunks/final 폴더를 확인해 보세요.


In [ ]:
import os
import json
import re
import vs_method

def normalize_and_ingest(file_path: str):
    # 1. DB 연결 및 스키마 세팅
    conn = vs_method.get_conn()
    vs_method.ensure_schema(conn)
    
    # ⚠️ 필요 시 기존 테이블을 완전히 비우고 새로 적재하려면 주석을 해제하세요.
    # vs_method.clear_table(conn)

    if not os.path.exists(file_path):
        print(f"❌ 파일을 찾을 수 없습니다: {file_path}")
        return

    chunks_buffer = []
    
    print(f"🔄 파일 읽기 및 메타데이터 정규화 시작: {file_path}")
    print("-" * 60)

    with open(file_path, 'r', encoding='utf-8') as f:
        for line_idx, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            
            try:
                data = json.loads(line)
            except json.JSONDecodeError:
                continue

            if "page_content" in data and data["page_content"].strip():
                content = data["page_content"]
                file_meta = data.get("metadata", {})

                # 🔍 텍스트 본문에서 '제OO조' 조항 번호 자동 파싱 시도 (없으면 지정값 사용)
                article_match = re.search(r"제(\d+조(?:의\d+)?)", content)
                detected_article = article_match.group(1) if article_match else None

                # 🎯 [올려주신 스펙 요구사항에 100% 맞춘 메타데이터 조립]
                # 기존 데이터에 값이 없으면 유실/에러를 막기 위해 지정된 디폴트 양식을 채웁니다.
                structured_metadata = {
                    "issue": file_meta.get("issue") or ["중개"],  # 반드시 리스트 배열 형태
                    "stage": file_meta.get("stage") or "pre",      # 소문자 pre/post
                    "article": file_meta.get("article") or detected_article or "N/A",
                    "case_no": file_meta.get("case_no") or None,   # 판례가 아니면 null
                    "doc_year": int(file_meta.get("doc_year") or 2026), # 정수형 변환
                    "law_name": file_meta.get("law_name") or "공인중개사법",
                    "authority": file_meta.get("authority") or "binding",
                    "doc_title": file_meta.get("doc_title") or "공인중개사법",
                    "source_id": file_meta.get("source_id") or f"001654_{line_idx:07d}",
                    "source_org": file_meta.get("source_org") or "국토교통부",
                    "chunk_index": int(file_meta.get("chunk_index") or 0), # 정수형
                    "source_type": file_meta.get("source_type") or "statute"
                }

                # 원본 pdf 페이지 정보 등이 있었다면 추가 보존 (선택사항)
                if "page" in file_meta:
                    structured_metadata["page"] = file_meta["page"]
                if "source" in file_meta:
                    structured_metadata["source"] = file_meta["source"]

                # 🧩 vs_method.ingest_not_chunks가 받아들이는 배치 구조로 패킹
                chunk_item = {
                    "content": content,
                    **structured_metadata  # 조립된 커스텀 메타데이터 통째로 합류
                }
                chunks_buffer.append(chunk_item)

    if not chunks_buffer:
        print("❌ 적재할 유효한 청크 데이터가 없습니다.")
        return

    # 2. 강제 매핑된 데이터 레이아웃을 들고 Supabase에 적재 실행
    print(f"📡 요청한 스펙의 메타데이터를 탑재하여 Supabase로 발송 중... (총 {len(chunks_buffer)}개)")
    
    # doc 인자에는 빈 껍데기를 주거나 기본 필수값만 주고, 
    # chunk_item 내부에 이미 정교하게 조립된 structured_metadata가 개별 적용되도록 넘깁니다.
    dummy_doc = {"source_type": "statute", "doc_title": "공인중개사법"}
    inserted_count = vs_method.ingest_not_chunks(conn, chunks_buffer, dummy_doc)
    
    print("-" * 60)
    print(f"🎉 [성공] 총 {inserted_count}개의 데이터가 요청하신 메타데이터 스펙과 일치하게 적재 완료되었습니다!")

# ──────────────────────────────────────────────
# 실행 구역
# ──────────────────────────────────────────────
if __name__ == "__main__":
    # 💡 아까 최종 청킹 완료한 .jsonl 파일의 실제 경로를 입력하세요.
    TARGET_JSONL = "../data/04_chunks/final/내_청킹_파일.jsonl"
    
    normalize_and_ingest(TARGET_JSONL)

In [6]:
import os
import json
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

def rechunk_json_file_dynamic(input_folder, output_folder, chunk_size=500, chunk_overlap=50):
    os.makedirs(output_folder, exist_ok=True)
    
    if not os.path.exists(input_folder):
        print(f"❌ 입력 경로를 찾을 수 없습니다: {input_folder}")
        return
        
    # 💡 .json과 .jsonl 둘 다 인식하도록 확장자 범위 확장
    files = [f for f in os.listdir(input_folder) if f.endswith('.json') or f.endswith('.jsonl')]
    print(f"📂 [디버깅] {input_folder} 폴더에서 검색된 파일 개수: {len(files)}개 ({files})")
    
    if not files:
        print("❌ 처리할 유효한 .json/.jsonl 파일이 없습니다.")
        return

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )

    for file_name in files:
        input_path = os.path.join(input_folder, file_name)
        output_path = os.path.join(output_folder, file_name)
        
        try:
            with open(input_path, 'r', encoding='utf-8') as infile:
                # 💡 JSONL 포맷일 경우를 대비한 유연한 로드 샌드박스
                content = infile.read().strip()
                if content.startswith('['):
                    original_data = json.loads(content)
                else:
                    # 한 줄씩 json 구조인 경우 (JSONL 대응 포팅)
                    original_data = [json.loads(line) for line in content.split('\n') if line.strip()]
            
            if isinstance(original_data, dict):
                original_data = [original_data]
                
            new_chunks_list = []
            print(f"🔄 {file_name} 파싱 시작... (총 아이템 수: {len(original_data)}개)")

            for line_idx, item in enumerate(original_data, 1):
                # 💡 안전한 딕셔너리 구조 추출로 조건문 완화
                if isinstance(item, list):
                    if len(item) == 0: continue
                    item = item[0]
                
                if not isinstance(item, dict):
                    continue

                base_content = item.get('page_content') or item.get('text') or item.get('content')
                if not base_content:
                    continue
                
                content_str = str(base_content)
                raw_metadata = item.get('metadata') if isinstance(item.get('metadata'), dict) else item

                # --- 🔍 본문 기반 동적 분석 매핑 ---
                if "판결" in content_str or "대법원" in content_str or "지방법원" in content_str:
                    source_type = "precedent"
                    source_org = raw_metadata.get("source_org") or "대법원"
                    authority = "binding"
                elif "신청인" in content_str or "피신청인" in content_str or "조정사례" in content_str:
                    source_type = "mediation_case"
                    source_org = raw_metadata.get("source_org") or "주택임대차분쟁조정위원회"
                    authority = "persuasive"
                else:
                    source_type = "statute"
                    source_org = raw_metadata.get("source_org") or "법제처"
                    authority = "binding"

                article_match = re.search(r"제\s*(\d+조(?:의\d+)?)", content_str)
                article = raw_metadata.get("article") or (article_match.group(1) if article_match else None)
                
                case_match = re.search(r"(\d{4}내?다\d+|\d{4}주조\d+|\d{4}고단\d+)", content_str)
                case_no = raw_metadata.get("case_no") or (case_match.group(1) if case_match else None)

                law_name = raw_metadata.get("law_name")
                if not law_name:
                    if "공인중개사법" in content_str: law_name = "공인중개사법"
                    elif "주택임대차보호법" in content_str or "주임법" in content_str: law_name = "주택임대차보호법"
                    elif "민법" in content_str: law_name = "민법"
                    else: law_name = "기타 법률/사례"

                issues = raw_metadata.get("issue") or raw_metadata.get("issues")
                if not issues:
                    detected_issues = []
                    if "중개" in content_str or "중개보수" in content_str or "복비" in content_str:
                        detected_issues.append("중개")
                    if "보증금" in content_str or "반환" in content_str:
                        detected_issues.append("보증금")
                    if "계약갱신" in content_str or "갱신요구" in content_str:
                        detected_issues.append("계약갱신")
                    if "수리" in content_str or "하자" in content_str or "유지상태" in content_str:
                        detected_issues.append("수리책임")
                    issues = detected_issues if detected_issues else ["일반"]

                stage = raw_metadata.get("stage")
                if not stage:
                    if "계약서 작성" in content_str or "특약" in content_str or "확인설명서" in content_str:
                        stage = "pre"
                    elif "해지" in content_str or "도과" in content_str or "연체" in content_str or "원상복구" in content_str:
                        stage = "post"
                    else:
                        stage = "both"

                split_texts = text_splitter.split_text(content_str)
                
                for idx, text in enumerate(split_texts):
                    standardized_metadata = {
                        "issue": issues,
                        "stage": stage,
                        "article": article,
                        "case_no": case_no,
                        "doc_year": int(raw_metadata.get("doc_year") or 2026),
                        "law_name": law_name,
                        "authority": authority,
                        "doc_title": raw_metadata.get("doc_title") or law_name,
                        "source_id": raw_metadata.get("source_id") or f"001654_{line_idx:04d}{idx:02d}",
                        "source_org": source_org,
                        "chunk_index": int(idx),
                        "source_type": source_type
                    }
                    
                    if "page" in raw_metadata: standardized_metadata["page"] = raw_metadata["page"]
                    if "source" in raw_metadata: standardized_metadata["source"] = raw_metadata["source"]

                    new_chunks_list.append({
                        "page_content": text,
                        "metadata": standardized_metadata
                    })
            
            # 💡 결과 파일 쓰기 및 로그 검증
            if new_chunks_list:
                with open(output_path, 'w', encoding='utf-8') as outfile:
                    json.dump(new_chunks_list, outfile, ensure_ascii=False, indent=4)
                print(f"✅ 저장 완료: {output_path} (생성된 청크: {len(new_chunks_list)}개)")
            else:
                print(f"⚠️ 경고: {file_name} 내부에서 유효한 텍스트 본문을 찾지 못해 저장할 내용이 없습니다.")
            
        except Exception as e:
            print(f"❌ {file_name} 처리 중 에러 발생: {e}")

# --- 실행부 ---
input_dir = "../data/04_chunks/semi"
output_dir = "../data/04_chunks/meta"

rechunk_json_file_dynamic(input_dir, output_dir, chunk_size=500, chunk_overlap=50)

📂 [디버깅] ../data/04_chunks/semi 폴더에서 검색된 파일 개수: 2개 (['tilnote_전세 사기 예방 등기부등본 확인법 초보 세입자 체크리스트_chunks.json', 'tilnote_2026년 전세 사기 예방 등기부등본 확인법 완벽 가이드_원본_chunks.json'])
🔄 tilnote_전세 사기 예방 등기부등본 확인법 초보 세입자 체크리스트_chunks.json 파싱 시작... (총 아이템 수: 60개)
✅ 저장 완료: ../data/04_chunks/meta/tilnote_전세 사기 예방 등기부등본 확인법 초보 세입자 체크리스트_chunks.json (생성된 청크: 60개)
🔄 tilnote_2026년 전세 사기 예방 등기부등본 확인법 완벽 가이드_원본_chunks.json 파싱 시작... (총 아이템 수: 59개)
✅ 저장 완료: ../data/04_chunks/meta/tilnote_2026년 전세 사기 예방 등기부등본 확인법 완벽 가이드_원본_chunks.json (생성된 청크: 59개)
